Native coroutine
A coroutine function defined with async def. You can delegate from a native
coroutine to another native coroutine using the await keyword, similar to how
classic coroutines use yield from. The async def statement always defines a
native coroutine, even if the await keyword is not used in its body. The await
keyword cannot be used outside of a native coroutine.

Asynchronous generator
A generator function defined with async def and using yield in its body. It
returns an asynchronous generator object that provides __anext__, a coroutine
method to retrieve the next item.

In [ ]:
import asyncio
import random

async def cook_dish(dish_name: str):
    """This is the 'probe' function."""
    # Simulating that different dishes take different amounts of time
    cook_time = random.uniform(1.0, 3.0) 
    
    # We yield control while it cooks!
    await asyncio.sleep(cook_time) 
    
    return dish_name, cook_time

async def main():
    # 1. The list of things we want to do (The website names)
    menu = ["Burger", "Fries", "Steak", "Salad", "Soup"]
    
    # 2. Writing the tickets (The 'coros' list)
    # We are NOT cooking yet. Just making a list of Coroutine Objects.
    tickets = [cook_dish(dish) for dish in menu]
    
    print("Handing all 5 tickets to the kitchen at once...")
    
    # 3. The Head Waiter (as_completed)
    # This fires all 5 tickets simultaneously.
    # It will loop 5 times, yielding whichever dish finishes FIRST.
    for finished_ticket in asyncio.as_completed(tickets):
        
        # We await the ticket to actually grab the food out of the kitchen
        dish, time_taken = await finished_ticket
        
        print(f"Served {dish} (Took {time_taken:.2f} seconds)")

if __name__ == '__main__':
    asyncio.run(main())

A normal for loop would make you wait for the Burger to finish before you even started cooking the Fries.

asyncio.as_completed() starts them all at once, and lets you process the results the millisecond they are ready, completely out of order.

await is an exit door from the function.

When you write await something(), the function literally stops existing in active memory. It is frozen in carbonite. The Event Loop takes the steering wheel back and drives other functions.

It only unfreezes when the Event Loop gets the signal that the something() is finished.

Python doesn't have a built-in async web downloader.
If you’ve ever written Python before, you might have used the requests library. But requests is Synchronous—if you use it, the Waiter stares at the wall until the web page loads, freezing the whole restaurant.

The Fix: uses HTTPX. It’s exactly like requests, but it has pagers (Awaitables) built-in!

In [ ]:
def download_many(cc_list: list[str]) -> int:
    return asyncio.run(supervisor(cc_list))


async def supervisor(cc_list: list[str]) -> int:
    async with AsyncClient() as client: # 1. The Phone Line
        
        # 2. Writing the Tickets
        to_do = [download_one(client, cc) for cc in sorted(cc_list)]
        
        # 3. Yelling the orders and waiting
        res = await asyncio.gather(*to_do)
        
        return len(res)

The Breakdown:

async with AsyncClient() as client:
Instead of dialing the phone, ordering one flag, and hanging up 20 separate times, this opens one massive, reusable delivery pipeline. The async with just means "Open this pipeline, use it, and politely close it when we are done."

to_do = [...]
Just like the last example, we are writing 20 order tickets (Coroutine Objects). No flags are downloading yet.

await asyncio.gather(*to_do)
This is the star of the show. Remember in the last example how as_completed served the food out of order as soon as it was ready?

gather is different. gather hands all 20 tickets to the kitchen at once, but the Waiter refuses to leave the kitchen until ALL 20 are completely finished.

When they are done, it returns them to you in the exact same order you asked for them.

In [ ]:
import asyncio
import httpx      # For the Async API/Network stuff
import aiofiles   # For the Async Hard Drive stuff

async def get_and_save_flag(country_code: str):
    # ==========================================
    # PART 1: THE API CALL (Talking to the Internet)
    # ==========================================
    url = f"https://example.com/flags/{country_code}.png"
    
    # syntax: async with opens a "connection pipeline" that automatically closes when done.
    async with httpx.AsyncClient() as client:
        print(f"[{country_code}] Requesting from internet...")
        
        # syntax: await pauses the function. The Event Loop goes to do other things 
        # until the internet finally sends the image data back.
        response = await client.get(url) 
        
        # We extract the raw image data (bytes) from the response
        image_data = response.content 

    # ==========================================
    # PART 2: SAVING THE FILE (Talking to the Hard Drive)
    # ==========================================
    file_name = f"{country_code}_flag.png"
    print(f"[{country_code}] Downloaded! Saving to hard drive...")

    # syntax: aiofiles.open allows the hard drive to write the file 
    # WITHOUT freezing the whole Python program.
    async with aiofiles.open(file_name, 'wb') as file:
        
        # syntax: await tells the Waiter to yield control while the physical 
        # hard drive spins and saves the image.
        await file.write(image_data)
        
    print(f"[{country_code}] Finished saving!")

# The Supervisor
async def main():
    # Create 3 tickets for 3 flags
    tickets = [
        get_and_save_flag("egypt"),
        get_and_save_flag("japan"),
        get_and_save_flag("brazil")
    ]
    # Fire them all at the exact same time
    await asyncio.gather(*tickets)

if __name__ == "__main__":
    asyncio.run(main())

Syntax Deep Dive: Why do we write it like this?
httpx.AsyncClient(): Older code used requests.get(). But requests is an old-school library that doesn't have "pagers" (it's not an Awaitable). httpx was built from the ground up for modern Python. It gives the Event Loop a pager.

async with: You know how if you open a file in Python, you have to remember to do file.close() so it doesn't corrupt? async with is a safety net. It says: "Open this internet connection, and the millisecond I am done with this indented block of code, close the connection safely for me."

'wb': This stands for "Write Bytes". Because an image isn't text, we have to tell the hard drive we are writing raw computer data (1s and 0s).

 If you ran the previous script in the real world on 1,000 flags, the target website would think you were a hacker (a DDoS attack), block your IP address, and your program would crash

Upgrade 1: The Bouncer (asyncio.Semaphore)
The Problem:
In the last script, if we asked for 1,000 flags, asyncio.gather would throw 1,000 tickets into the kitchen at the exact same millisecond. The internet server would panic and give you a 503 Service Temporarily Unavailable error.

The Fix:


The Translation:
A Semaphore is the Bouncer at the club.

When you set DEFAULT_CONCUR_REQ = 5, you tell the Bouncer: "Only let 5 Waiters go to the internet at a time."

When a Waiter hits the line async with semaphore:, they are asking the Bouncer for permission to enter.

If there are already 5 Waiters out there, the 6th Waiter awaits at the door. He yields control until one of the first 5 Waiters comes back and gives up their VIP pass.

This keeps your program polite and prevents the server from blocking you.

In [ ]:
async with semaphore:
    image = await get_flag(client, base_url, cc)

In [ ]:
# Upgrade 2
# this upgrade to handle error of not found
except httpx.HTTPStatusError as exc:
    if res.status_code == HTTPStatus.NOT_FOUND:
        status = DownloadStatus.NOT_FOUND

pgrade 3: The Delivery Boy (asyncio.to_thread)
This is the exact fix for the "Gotcha" we talked about earlier!

The Problem (Review):
Saving a file to a hard drive (save_flag) is a Synchronous, Blocking action. If the main Waiter (the Event Loop) carries the file to the hard drive, the whole restaurant freezes.

The Fix:

The Translation:
Because the Python developers know that saving files asynchronously is a pain, they built a shortcut in Python 3.9 called to_thread.

This is the Waiter hiring a Delivery Boy (a Thread).

The Waiter gets the image from the internet.

Instead of walking to the freezer himself, he hands the image to the Delivery Boy and says: "Hey, go put this on the hard drive. I am going to yield (await) and go help Table 4. Let me know when you're done."

The Delivery Boy (who is on a separate thread) walks to the freezer. The restaurant does not freeze.

When the Delivery Boy finishes, the Waiter gets a pager buzz and continues.

In [ ]:
await asyncio.to_thread(save_flag, image, f'{cc}.gif')

### Upgrade 2 discussion:

Method A: return_exceptions=True (Catching Flying Plates)
If you don't use try/except inside the function, and instead rely on gather(..., return_exceptions=True), here is what happens:

The Waiter goes to the kitchen to get the "Atlantis" flag.

The kitchen says, "We don't have that!"

The Waiter panics, screams, and literally throws a plate at the Manager (raises an Exception).

Because the Manager has return_exceptions=True, he is wearing a helmet. He catches the broken plate and puts it in the pile of finished orders.

The Result: When gather finishes, the Manager looks at the final list of results. It looks like this:
["USA", "Brazil", <HTTPStatusError: 404 Found Not>, "Japan"]

The Manager now has a messy list full of good data mixed with literal Error Objects. He has to write extra code to loop through and check, "Is this item a string, or is it a broken plate?"

Method B: Upgrade 2 try/except (The Professional Waiter)
If you handle the error inside the function (Upgrade 2), here is what happens:

The Waiter goes to the kitchen to get the "Atlantis" flag.

The kitchen says, "We don't have that! (404 Error)"

The Waiter says, "No problem, I expected this." He takes out his notepad, writes down DownloadStatus.NOT_FOUND, and hands that neat receipt to the Manager.

The Result: When gather finishes, the Manager gets a perfectly clean list:
[<Status.OK>, <Status.OK>, <Status.NOT_FOUND>, <Status.OK>]

No broken plates. No panic. Just clean, organized data.

The 2 Big Reasons We Used Upgrade 2 Here
1. We only want to ignore expected errors.
The author explicitly checks: if res.status_code == HTTPStatus.NOT_FOUND:
He is saying, "If the flag doesn't exist, that's fine. But if the website is completely offline (500 Error), or my internet is disconnected, I WANT the program to crash loudly so I can fix it."
If you use return_exceptions=True, it sweeps all errors under the rug, which makes debugging terrible because it hides real bugs.

2. We need a specific return type.
The author's function is designed to return a DownloadStatus (OK, NOT_FOUND, or ERROR). If you let gather catch the exception, it returns a literal Python Exception object, which ruins the strict type-checking of the program.


In [ ]:
from pathlib import Path
from unicodedata import name
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from charindex import InvertedIndex

# 1. PATH RESOLUTION
# Using pathlib's overloaded '/' operator to elegantly build file paths.
# This creates an absolute path to a 'static' folder sitting right next to this script.
STATIC_PATH = Path(__file__).parent.absolute() / 'static'

# 2. APP INITIALIZATION
# This defines the ASGI application. 
# The parameters provided here act as metadata for the autogenerated /docs (OpenAPI/Swagger) page.
app = FastAPI(
    title='Mojifinder Web',
    description='Search for Unicode characters by name.',
)

# 3. PYDANTIC SCHEMA
# This enforces type hints at runtime for data validation.
# It guarantees that the JSON response will always contain 'char' and 'name' fields.
class CharName(BaseModel):
    char: str
    name: str

def init(app):
    # 4. STATE MANAGEMENT
    # Build the inverted index and load the static HTML form into memory.
    # Storing them in `app.state` makes them accessible throughout the app lifecycle.
    app.state.index = InvertedIndex()
    app.state.form = (STATIC_PATH / 'form.html').read_text()

# 5. INITIALIZE ON LOAD
# This runs when the ASGI server (like uvicorn) loads this module.
init(app)

# 6. THE SEARCH ENDPOINT (API)
# response_model=list[CharName] tells FastAPI to format the output using the Pydantic schema above.
@app.get('/search', response_model=list[CharName])
async def search(q: str):
    """
    Because 'q' is in the signature but not the route path ('/search'), 
    FastAPI expects it as an HTTP query string parameter (e.g., /search?q=cat).
    If a user doesn't provide 'q', FastAPI automatically returns a 422 error.
    """
    chars = sorted(app.state.index.search(q))
    
    # We return a generator of dictionaries. FastAPI takes this, validates it 
    # against the `response_model`, and serializes it into the final JSON list.
    return ({'char': c, 'name': name(c)} for c in chars)

# 7. THE ROOT ENDPOINT (FRONTEND)
# include_in_schema=False hides this HTML route from the autogenerated OpenAPI JSON docs.
@app.get('/', response_class=HTMLResponse, include_in_schema=False)
def form():
    # Notice this is a regular `def`, not `async def`. 
    # FastAPI can seamlessly handle standard synchronous functions alongside async ones.
    return app.state.form

# 8. NO MAIN FUNCTION
# There is no `if __name__ == '__main__':` block. This module is driven entirely 
# by the ASGI server (uvicorn), not executed directly as a standalone script.

IndexError: list index out of range

What is TCP?
Before we look at the code, let's talk about TCP (Transmission Control Protocol).

Imagine you are trying to send a 100-page book to a friend in the mail, but you are only allowed to mail one page at a time.

If you just drop 100 separate envelopes in a mailbox (this is what UDP does), some might get lost, and they will almost certainly arrive out of order. Your friend would have a chaotic mess.

TCP is like a certified, numbered delivery service. It guarantees three things:

Connection: It establishes a dedicated line (like a phone call) before sending anything.

Order: It numbers every packet of data. If page 4 arrives before page 3, TCP holds onto page 4 and waits for page 3 before handing the data to your application.

Reliability: If page 7 gets lost in the mail, your friend's computer automatically tells your computer, "Hey, I never got page 7," and your computer resends it.

TCP is the backbone of the internet. It's the "raw" pipe that higher-level protocols (like HTTP for websites) use to make sure no data goes missing.